## Introduction

Recent AI systems have become capable of carrying out some multi-step tasks with limited human guidance. A research assistant can look up a topic, read what it finds, and decide what to search for next. We call systems that choose actions and respond to their results in pursuit of a goal **AI agents**, or **agentic systems**[^novelty]. Our primary interest are agents with a [(reasoning) large language model (LLM)]{.mark} acting as the *controller*. This works out because many of the resources an agent works with are already accessible through text. The model can read instructions and documents, write a SQL query, and use the query results to choose its next action. Most importantly, agents interact with computing environments through the terminal, where it can access APIs and even write scripts to run programs. Furthermore, LLMs are not strictly confined to text input&mdash;modern multimodal models are able to ingest other modalities such as audio, video, and images.

Agentic systems are architected such that the LLM itself dynamically reasons about and controls the sequence of actions. The model selects tools, evaluates outcomes, and iterates until a task is complete, making the path **autonomous**, **non-deterministic** and **goal-oriented**. We will discuss various patterns that takes advantage of this flexibility as well as create safeguards around it.[^tradeoff] This contrasts with more traditional uses of LLMs as components in a workflow setup (e.g. document translation + summarization pipeline).

[^novelty]: The idea of autonomous agents is not exactly new; the novelty lies in that LLM-powered agents does this using an [informal]{.underline} and [inexact language]{.underline} (e.g. the [English language](https://x.com/karpathy/status/1617979122625712128)), but in a programmable, highly performant, and stochastic manner (creative?) &mdash; very much like humans.

[^tradeoff]: A key engineering trade-off is that this autonomy typically [increases latency]{.underline} and [cost]{.underline} compared to a single LLM call (e.g. in a fixed workflow), making architectural choice a critical design decision. For example, a program should always have an **upper bound** on the total number of tokens processed within a fixed time period. By definition, autonomy also makes these systems less predictable and strong alignment and guardrails are required to [ensure safety](https://en.wikipedia.org/wiki/AI_safety).

<!-- ![**AI Agent building block.** Core LLM as engine for reasoning, planning, and tool calling. Agents follow goals and interact with their environment (external systems, such as other agents). This is not exactly new, the novelty lies in that AI agents does this using an [informal]{.underline} and [inexact language]{.underline} (e.g. the [English language][agents-caption-english]), but in a highly performant and stochastic manner &mdash; very much like humans.](../images/post/ai-agents-foundations-499cc80d835d406d8a6f4a28a519c493.png){#fig-augmented-llm} -->

[agents-caption-english]: https://x.com/karpathy/status/1617979122625712128


## Limitations of LLMs

A language model can produce plausible claims that are false or unsupported, a failure known as **hallucination**. Its training cutoff can leave gaps in what it knows about recent events. When a task spans several model calls, the surrounding system carries forward relevant history or retrieves it when needed. Performance also varies by task and input type, so [model choice is nontrivial]{.mark}: it is **task dependent**, which adds friction when the agent has to work on varied tasks.


### Hallucination

Pretraining compresses patterns from a large corpus into a model’s parameters as it learns to predict each token from the preceding text. The resulting representations help the model generalize across examples. A fact that appears only rarely in training can be harder to learn reliably [@kalai2024calibrated]. When the evidence is sparse or conflicting, the model can assign probability to several plausible continuations. A familiar association can also give an incorrect detail high probability.[^quality] 

Suppose the previous champion loses a tournament held after the model’s training cutoff. With little evidence about the result in the prompt, the model might name that champion as the winner. That name becomes part of the prefix for subsequent predictions, so the model may generate a score or match summary that fits the wrong winner. The answer can read fluently and hang together from one sentence to the next, making the factual error hard to spot.

[^quality]: Coverage and data quality both matter. Repeated misinformation and biased associations can influence the patterns a model learns. In a standalone model call, the model draws on those learned patterns and the supplied context. A vague prompt can leave several interpretations open, and conflicting evidence in the context can compound the problem. [McKenna et al. on learned associations and inference errors](https://aclanthology.org/2023.findings-emnlp.182/)


### Performance gaps

Another issue that LLMs have is performance gaps between different task categories, perhaps due to gaps between training data quality and availability (e.g. [GPQA](https://arxiv.org/abs/2311.12022), [MMLU](https://en.wikipedia.org/wiki/MMLU), [HumanEval](https://github.com/openai/human-eval), [MATH](https://huggingface.co/datasets/nlile/hendrycks-MATH-benchmark)). See @fig-llm-stats from [llm-stats](https://llm-stats.com/benchmarks). This adds  difficulty in choosing a model for a given task. The hope is that by equipping the LLMs with tools, the skill gap between same-sized models becomes insignificant. Another approach is to dynamically change models or **triaging**[^humans] by using an initial model for categorizing the problem, then sending it to the appropriate specialized LLM.

[^humans]: A lot of agentic design pattern is inspired by how humans organize to solve tasks. For example, the triaging pattern is used in a hospital setting to deal with incoming patients with unknown afflictions while having doctors with deep specializations.

![**Model performance on various tasks.** Source (retrieved Oct 2025): [llm-stats][agents-caption-llm-stats]](../images/post/ai-agents-foundations-b2396ca962384a799e6315f64d933590.png){#fig-llm-stats}

### Prompting

Finally **prompt sizes** are limited finite resources. On Oct 2025, Llama 4 has an industry-leading context window of 10M, while Wikipedia consists of ~38B tokens. Moreover, as the context length grows, reasoning capabilities of LLM generally drops (@fig-needle_question_sim_arxiv). This motivates the practice of **context engineering** which refers to the set of strategies for curating and maintaining the optimal set of tokens (e.g. instructions) during LLM inference, including other information not included in the prompts (e.g. data or facts obtained via retrieval / tool calls).

![**Decreasing performance for needle in a haystack setup.** The classic Needle in a Haystack task involves placing a random fact (the 'needle') in the middle of a long context window (the 'haystack'), then asking the model about that fact. Similarity refers to the similarity of the question to the needle. The degradation are observed for both long-context and usual models as well as across performance tiers.
](../images/post/ai-agents-foundations-12e2204906c0406080ab369c55a0727c.png){#fig-needle_question_sim_arxiv}

![**Prompt engineering vs. Context engineering.** Context engineering involves systematically adding invoke-specific information for the LLM to consume. This typically includes system instructions (rules, behavior, e.g. tone), tool definitions, retrieved documents (RAG), conversation history or summaries of, memory, as well as structured data.
](../images/post/ai-agents-foundations-19651dc0ebc648bc8226753bef3b8af5.png){#fig-context_engineering}

Context engineering also involves where evidence appears. *Lost in the Middle* tested this by moving the answer-bearing document through the same set of retrieved passages. In @fig-lost-in-the-middle, accuracy is highest at the start, falls in the middle, and recovers toward the end [@liu2023lostmiddlelanguagemodels].

[agents-caption-llm-stats]: https://llm-stats.com/benchmarks


![**Position sensitivity in multi-document question answering.** GPT-3.5-Turbo-0613 receives 20 documents (about 4K tokens), one of which contains the answer. The x-axis gives that document’s position. The dashed line is the model’s closed-book accuracy with no retrieved documents. Figure 1 of [@liu2023lostmiddlelanguagemodels].](../images/post/ai-agents-foundations-9d75d21150c44736aa1a222b6b447679.png){#fig-lost-in-the-middle fig-alt="Accuracy for GPT-3.5-Turbo-0613 falls from about 76 percent with the answer document first to about 54 percent near the middle, then recovers to about 63 percent at the end. The closed-book baseline is about 56 percent."}


## Agent harnesses {#sec-agent-harnesses}

An **agent harness** runs the loop between the model and its tools. This essentially sets up the environment for the agent. More precisely, it prepares the context for each model call, routes tool requests, and makes the results available to the next call. The ReAct exercise uses a small harness that runs Wikipedia searches, returns the excerpts as observations, and stops after at most five model calls. [ReAct exercise](#exercise-2.-a-react-search-trace)

### Separating the loop from execution and storage

[Anthropic’s Managed Agents architecture](https://www.anthropic.com/research/managed-agents) runs the harness separately from the **session log** and the **sandbox**. The session log keeps a durable record of events such as messages, tool calls, and results. Tools perform the requested actions, using a sandbox for operations such as running shell commands and editing files. This separation changes what happens when a component fails. A replacement harness can read the session log and resume from the last recorded event. If a sandbox goes offline during a tool call, the harness can return an error to the model, which can retry or choose another action. 

```{mermaid}
%%| label: fig-agent-harness-loop
%%| fig-cap: "One tool call in an agent loop with a session log."
---
config:
    themeCSS: |
        rect.actor { fill: #fff2a8 !important; fill-opacity: 1; stroke: #000 !important; stroke-opacity: 1; }
        text, tspan { fill: #000 !important; fill-opacity: 1; opacity: 1; }
        .messageLine0, .messageLine1 { stroke: #000 !important; stroke-opacity: 1; opacity: 1; }
        marker, marker path { fill: #000 !important; stroke: #000 !important; opacity: 1; }
---
sequenceDiagram
    accTitle: One tool call in an agent loop
    accDescr: The harness reads past events, calls the model, runs a requested tool, and records the outcome before calling the model again.
    participant S as Session log
    participant H as Harness
    participant M as Model
    participant T as Tool or sandbox
    H->>S: Read prior events
    S-->>H: Messages and results
    H->>M: Context and tools
    M-->>H: Tool request
    H->>T: Execute request
    T-->>H: Result or error
    H->>S: Record the outcome
    H->>M: Continue with the observation
```

### Session history and model context

The session can hold more history than fits in a model call. The harness can read a slice of the log and summarize it before sending it to the model. For example, a debugging agent might get a short summary of a failed build in its next call. If it needs the lines leading up to the error, the harness can fetch the original output from the log. Keeping the original events makes that later lookup possible even after the model’s context has been condensed.


## Foundational papers

The rapid publication of **ReAct** [@ReAct2023], which is based on **Chain-of-Thought Prompting** [@cot], **Toolformer** [@toolformer], and **Generative Agents** [@genagents] within a few months of each other in early 2023 marks the period when LLMs transitioned from powerful stateless tools to the reasoning engines for autonomous, tool-using agents that we have today. These key papers laid the foundations for the capabilities of agentic systems.

### Chain-of-Thought [@cot]

**Chain-of-Thought Prompting** (CoT) is a technique where a non-reasoning model is prompted by few-shot examples of step-by-step reasoning steps that lead to the final output (@fig-cot). This resulted in SOTA performance (at the time) on the [GSM8K benchmark](https://github.com/openai/grade-school-math) consisting of grade school math word problems, from 18% to 57% solve rate for the PaLM 540B *just by prompting* (surpassing even a fine-tuned GPT3 at 33%) &mdash; the authors benchmarked the technique with other tasks such as robot instruction and commonsense reasoning, likewise getting significant improvement.

These results were interpreted as LLM's having **emergent abilities** that can be *unlocked* if just asked to do so explicitly. It also opened the paradigm that fine-tuning was not necessary to get SOTA performance. Finally, since CoT only involves a change in prompt design, it was easily applicable to every LLM use-case.

![**Figure 1 of** [@cot]. **Few-shot and Zero-shot CoT.** Chain-of-thought prompting enables LLMs to tackle complex arithmetic, commonsense, and symbolic reasoning tasks just by a [simple trick]{.underline}. Chain-of-thought reasoning processes are highlighted.](../images/post/ai-agents-foundations-62bc825947a840bdafdb8ae3875cb538.png){#fig-cot}

### ReAct [@ReAct2023]

CoT (the "reasoning" part) was further augmented by **acting** (e.g. action
plan generation) in this paper. This paper 
explored the use of LLMs to generate both reasoning traces and task-specific actions in an interleaved manner resulting in greater synergy: reasoning traces help the model to play around action plans (e.g. handle exceptions), while actions allow it to interface with external sources (knowledge bases, Wikipedia) thereby gathering additional information (@fig-react-fig1). 

Performing actions resulted in improved **groundedness** and **trustworthiness** for the system: authors report 0% of ReAct failures are hallucinations, compared to 56% for CoT, instead 47% failures for ReAct are due to wrong reasoning traces including repetitive steps. In the paper, ReAct shows strong generalization to new task instances with only 1-6 **few-show examples** (i.e. manually composed ReAct-format trajectories, no fine-tuning[^react-finetuning]), consistently outperforming only reasoning or only acting baselines across different domains. 

![**Figure 1 of** [@ReAct2023]. The combination of reasoning and acting allows the agent to solve the task vs. reason-only and act-only approaches. ReAct outperforms imitation and reinforcement learning methods on two interactive decision making benchmarks (ALFWorld and WebShop) by an absolute success rate of 34% and 10% respectively, while being prompted with only one or two in-context examples.](../images/post/ai-agents-foundations-ae112ff5b63e4ba5bf76282cb3dd9733.png){#fig-react-fig1}

[^react-finetuning]: The authors attempted to fine-tune a smaller model PaLM-8/62B via **distilling** ReAct-format reasoning traces from PaLM-540B. That is, instead of conditioning the model via prompts, the reasoning traces are baked into the generating process. The resulting trained model outperformed all 540B pure-prompting methods. The authors believe that finetuning with more human-written data may unleash the full potention of ReAct.

### Toolformer [@toolformer] 

The next paper focused on **tool calling**. At this point, LLMs have been shown to be proficient in few-shot prompting. But it still struggles with basic tasks like arithmetic or [counting r's in "strawberry"](https://techcrunch.com/2024/08/27/why-ai-cant-spell-strawberry/). Moreover, it's knowledge is frozen after training cutoff, and as such is outdated by the time it gets to the user. All these issues are solved if the model is able to use tools. 

In the Toolformer paper, it is shown how a language model can be **trained** for **API calling** without fundamentally changing the model's architecture or training objective, and without a specialized dataset. The model learns to decide *which* API to call, *when* to call it, *what* arguments to pass, and *how* to best incorporate the results into the token generation process. The goal is to insert API calls into text generation. This were done in the ff. steps:

1. **Self-supervised data generation.** The key innovation of the paper is that the API calls are generated without needing human labelers. The model itself to generate potential API calls from text. The language model is exposed to few-shot examples of API calls inserted in [helpful places]{.underline} (the *when*). From Appendix A.2 of the paper, we see the prompts used. Here's one for the calculator tool:

    ```
    Your task is to add calls to a Calculator API to a piece of text. The calls sho
    uld help you get information required to complete the text. You can call the AP
    I by writing "❬Calculator(expression)❭" where "expression" is the expression to
     be computed. Here are some examples of API calls:
    
    Input:  The number in the next term is 18 + 12 x 3 = 54.
    Output: The number in the next term is 18 + 12 x 3 = ❬Calculator(18 + 12 *3)❭ 5
    4.

    Input:  A total of 252 qualifying matches were played, and 723 goals were score
    d (an average of 2.87 per match). This is three times less than the 2169 goals 
    last year.
    Output: A total of 252 qualifying matches were played, and 723 goals were score
    d (an average of ❬Calculator(723 / 252)❭ 2.87 per match). This is twenty goals 
    more than the ❬Calculator(723 - 20)❭ 703 goals last year.
    
    Input: {x}
    Output:
    ```

    Then, for an input sequence $\mathbf{x}$ we iterate over positions $i$ in the text and save those where 
    the model conditioned on $[\Phi(\mathbf{x}), \mathbf{x}_{<i}]$ assigns a high probability of an API call $\boldsymbol{\langle}$ as 
    next token[^sampling_threshold]. Here $\Phi(\mathbf{x})$ is the accompanying tool call prompt (see above) &mdash; so that the model sees the entire input, giving it hints on what the expected
    API call output should be.
    Next, for each such position we get a number of API args $\mathbf{c}^1_i, \ldots, \mathbf{c}^m_i$ (one tool with different arguments) by generating text given $[\Phi(\mathbf{x}), \mathbf{x}_{<i}, \boldsymbol{\langle}]$ until the model outputs $\boldsymbol{\rangle}$ as EOS token (examples that do not generate this are discarded). Note that [no actual execution]{.underline} is performed so far. We only want a sample of positions along with API args for that position for each input $\mathbf{x}.$[^phase1]

    ![**Examples of API calls and responses.** The tools are designed to take natural language as input. The tool specification influence the design of few-shot prompts $\Phi.$ See paper for details (e.g. the MT tool serves [NLLB-600M][agents-caption-nllb] a multi-lingual machine translation model for 200 languages with [fastText][agents-caption-nllb-2] to identify the source language).](../images/post/ai-agents-foundations-d6181e2a6876400e87493cc70b681092.png)

    In the next phase, we perform the executions of the sampled API args $\mathbf{c}^1_i, \ldots, \mathbf{c}^m_i$. For $l = 1, \ldots, m$, the response needs to be a single text sequence $\mathbf{r}_i^l$ (this includes error responses). Finally, we filter out the sample API calls by evaluating to see if the tokens of $\mathbf{x}$ after $i$ is assigned a significantly lower loss (vs. some error threshold[^api_threshold] $\tau_f$) vs. the generation [without]{.underline} tool call[^actual]. 
    The API calls with their execution results are then interleaved in the text to get $\mathbf{x}^* = [\mathbf{x}_{<i}, \boldsymbol{\langle}\mathbf{c}_i \rightarrow \mathbf{r}_i \boldsymbol{\rangle}, \mathbf{x}_{\geq i} ].$
    Doing this process of candidate generation, execution, and filtering (@fig-toolformer) to every text in the original dataset yields the final, curated dataset used for fine-tuning. 
    Some examples in the fine-tuning dataset[^ft-format]:

    ```
    The name derives from “la tortuga”, the Spanish word for ❬MT(“tortuga”) → turtle❭ turtle.
    Out of 1400 participants, 400 (or ❬Calculator(400 / 1400) → 0.29❭ 29%) passed the test. 
    ```
    
2. **Standard fine-tuning & modified inference.** The language model is then trained on the augmented tool-use dataset using standard techniques. Note that the output of the API calls are already in the dataset so we don't have to perform any execution at this stage. However, during inference the engine is designed such that whenever the token `→` is detected, the decoding is paused to execute the API call and the generation continues with the result and the EOS `❭` added to the context. The authors used greedy decoding with `❬` selected if it's in the top-*k* of candidate tokens to encourage tool-calling, with limitations on the amount of tool calls per input. The model is expected to perform tool-calling in a **zero-shot** setup (i.e. no examples of tool-calling are needed in its prompt.)

![**Figure 2 of** [@toolformer]. A language modeling dataset is augmented with API calls. Instead of just predicting "the Steel City", an API call is created as an in-between target.](../images/post/ai-agents-foundations-851c5632050d4d76a07bcbea8a50c420.png){#fig-toolformer}

[^sampling_threshold]: The top-$k$ positions in the text $\mathbf{x}$ with next-token probability of an API call that exceeds $\tau_s = 0.05$ or 5%. For large datasets, $k=5$ and $m=5$ were used. For smaller, this is increased to $k=20$, $m=10$, and $\tau_s = 0.$

[^phase1]: This phase focuses on gathering candidate data that can potentially teach a model on 
*when* to call and API and *which* API arguments to call. Not so much how the actual return values of the 
call affects future generation.

[^api_threshold]: The size of the generated dataset decrease rapidly as $\tau_f$ increases (see Table 2 in the paper).

[^actual]: Suppose the API call happens at index $i$ in a sequence $\mathbf{x}$, the model is conditioned
on $[\mathbf{z}, \mathbf{x}_{<i}]$ and a weighted cross-entropy of the ground truth sequence $\mathbf{x}_{\geq i}$ is calculated, with the prefix $\mathbf{z}$ being either (1) FULL `❬f(args) → r❭` where $\mathbf{c}_i =$ `f(args)` and $\mathbf{r}_i =$ `r`, (2) MASKED RESULT `❬f(args) → ❭` and (3) NO API call (i.e. the usual generation). A sample API call is kept only if the weighted cross-entropy of $\mathbf{x}_{\geq i}$ (1) is the minimum compared to (2) and (3) by at least $\tau_f.$ Machine Translation (`MT`) tool example:
```
(1) [❬MT(“tortuga”) → turtle❭ The name derives from “la tortuga”, the Spanish word for] turtle.
(2) [❬MT(“tortuga”) → ❭ The name derives from “la tortuga”, the Spanish word for] turtle.
(3) [The name derives from “la tortuga”, the Spanish word for] turtle.
```
This intuitively means that the act of calling a particular API is not only useful, but its result is also useful. Here the API call $\mathbf{z}$ is added as a prefix in getting $p_\textsf{model}(\mathbf{x}_{\geq i} \mid [\mathbf{z}, \mathbf{x}_{<i}])$ is so that the complete sequence $[\mathbf{z}, \mathbf{x}_{<i}, \mathbf{x}_{\geq i}]$ remains in-distribution w.r.t. the pretraining data.
<br><br>
Finally, the cross-entropy weights $w_t \geq 0$ where $t = j - i$ decay for increasing $t.$ 
The authors used $w_t = \max(0, 1 - 0.2 \cdot t)$ which means only 6 next tokens contribute to the loss. 
These are normalized over the sequence, i.e. the actual weights
used are $\tilde{w}_t = w_t / \sum_{t^\prime} w_{t^\prime}.$
Using decaying weights ensures that API calls happen close to where
the information provided by the API is actually helpful for the model. 

[^ft-format]: This format is exactly how API calls during the generation process are performed. In the actual paper, `[`, `]`, and `->` are used so that everything works without modifying the existing vocabulary. 

[agents-caption-nllb]: https://huggingface.co/facebook/nllb-200-distilled-600M
[agents-caption-nllb-2]: https://github.com/facebookresearch/fastText


The self-supervised data generation process relies on base model already possessing a rich, implicit knowledge of tools and their purposes from its pre-training (humans use tools intuitively, and tool use follows causal and physical rules, all of these reflect in recorded language data, so it's likely that LLMs also learn it from the statistics of the data). Assuming this is true, what the LLM then lacks is the 'grammar' for tool calls[^grammar_tool_call]. The fine-tuning phase solved this by forging a new, high-probability pathway into its output generation (i.e. using the tool call token $\text{❬}$ and forming proper arguments) in contexts where information has to be identified from external sources (@fig-toolformer-evals).

![**Performance of Toolformer in downstream tasks.** Toolformer was evaluated on a variety of downstream tasks. A zero-shot setup is used where no tool-calling examples were provided in the prompt. Greedy decoding was used with an API call triggered when $\text{❬}$ is found in the top-10 next token candidates. Toolformer with [tool-calling disabled]{.underline} was also evaluated to ensure that the base performance did not degrade (i.e. forcing $p(\text{❬}) = 0$). GPT-J + CC refers to [GPT-J][agents-caption-gpt-j] (6B) fine-tuned on a subset of [Common Crawl][agents-caption-gpt-j-2], and Toolformer is the same GPT-J fine-tuned on CC$^*$ (i.e. Common Crawl augmented with API calling discussed). Interestingly, Toolformer with API calling disabled performs better than GPT-J + CC &mdash; it is surmised that being exposed more API calls and their results during training improved its own reasoning abilities[^pure-lm-perf]. Effect of tool calling is also emphasized by beating GPT-3 (175B) a significantly larger model in these tasks.](../images/post/ai-agents-foundations-95627eec362443d9955bdad6dea56113.png){#fig-toolformer-evals}

[^grammar_tool_call]: Toolformer translated the tool-use problem into a pure text completion problem, and showed it was possible. Without having read the paper, I would think an RL-based approach would be required to successfully solve this. However, from the empirical analysis of scaling laws in the paper (**Figure 4**), it was shown that Toolformer ([GPT-2]{.underline} fine-tune) only beats Toolformer (disabled &mdash; i.e. forcing $p(\text{❬}) = 0$) past **775M parameters**, implying that tool-calling capabilities started to emerge once language ability was sufficiently high. The scaling laws also show that the gap between Toolformer and Toolformer (disabled) does not decrease past the parameter count where the model learns tool-calling.

[^pure-lm-perf]: How about performance on pure language modeling (e.g. "Write me a story")? **Table 8** of the paper shows base language modeling performance of the models in terms of perplexity. It is shown that Toolformer (disabled) (+ CC$^*$) did not degrade relative to GPT-J + CC which is the valid comparison. It did degrade in [WikiText](https://huggingface.co/datasets/Salesforce/wikitext) which is fine since GPT-J + CC also did, so this may be just due to distribution shift.

[agents-caption-gpt-j]: https://en.wikipedia.org/wiki/GPT-J
[agents-caption-gpt-j-2]: https://en.wikipedia.org/wiki/Common_Crawl


### Generative agents [@genagents]

Finally, the last paper *Generative Agents: Interactive Simulacra of Human Behavior* dives into the ability of agents to simulate believable human behavior. This highlights the ability of LLM-based agents to interact with their **environment** which includes other agents. Hence, providing an example of a working multi-agent architecture. The agents in this paper are characterized as being able to **remember**, **reflect**, and **plan** based on growing memory and casading social dynamics. In the paper, a sandbox environment are filled with 25 LLM-based agents that are shown to demonstrate believable human behavior (@fig-genagents). 

![**Figure 1 of** [@genagents]. Generative agents socialize, chill out, talk about politics & news, and perform routines. Moreover they have been observed to perform longer-term planning and coordinate with other agents in their plans.](../images/post/ai-agents-foundations-d8a639eee906406887121f7e8f3d263a.png){#fig-genagents}

**Generative agents.** Each agent is [seeded]{.underline} with a paragraph in natural language that depict their identity (including their occupation and relationship with other agents). Each `;`-delimited phrase is entered into the agent’s **initial memory** as memories at the start of the simulation (later we will discuss the memory stream in more detail). 

Generative agents operate in
an action loop where, at each time step, they perceive the world
around them and perform actions, or talk to other agents. 
Consider the agent John Lin. John Lin
interacts with their world in several key ways:

1. **Action.** At each time step, an agent outputs a natural language statement describing its current action ("John Lin is researching the local mayor election"). This includes interaction with their environment (e.g. "closet is being used to select clothes for the day"). Agents can also move between different locations in the map. 

2. **Observation.** Agents can record the state of their local environment in their memory ("refrigerator is idle"). This includes other agents: "Adam Smith is discussing the topic with the other participants". 

3. **Dialogue.** Agents converse as they interact with each other. Agents' dialogue are generated by conditioning 
on their memory of each other and the current summary status of the agent, as well as their observation of the 
other agent whom they want to engage in a conversation with:

    ```
    [Agent's Summary Description:]
    It is February 13, 2023, 4:56 pm. 
    John Lin's status: John is back home early from work. 

    [Observation:] 
    John saw Eddy taking a short walk
    around his workplace.

    [Summary of relevant context from John's memory:]
    Eddy Lin is John's Lin's son. Eddy Lin has been
    working on a music composition for his class. Eddy
    Lin likes to walk around the garden when he is
    thinking about or listening to music.
    John is asking Eddy about his music composition
    project. 

    What would he say to Eddy?
    ```

The continuation of this dialogue is generated using
the same mechanism until one of the two agents decides to end the
dialogue. Below (in the *reacting* discussion) it is discussed in more detail how inter-agent dialogues are triggered.

:::{.callout-tip}
## Emergent behavior
These interactions result in emergent behavior such as [information diffusion]{.underline} and [coordination]{.underline} (planning an event involving multiple agents helping each other out and arriving on time at a specific location).

:::


**Memory and retrieval.** Everything that an agent experiences is recorded and reasoned over as a natural language description, 
allowing the architecture to leverage a LLM. 
Generative agents take their current environment and past experiences as [input]{.underline} and generate behavior as output.
Underlying this behavior is a novel agent architecture that combines a large language model with mechanisms
for synthesizing and retrieving relevant information to condition
the language model’s output.

- **Memory stream.** This is a list of 
memory objects for each agent that contains a timestamp created, timestamp of last access (which defaults to created), 
and a natural language description. 
Observations, i.e. event directly perceived by agent (e.g. behavior of self, other, non-agent objects)
are stored in memory. Conversation between agents are summarized in the memory stream.

- **Retrieval function.** For a query $Q$, each memory $m$ receives three scores.
    *Recency* is $0.995^{\Delta t_m}$, where $\Delta t_m$ is the number of
    simulated hours since the memory was last accessed. *Relevance* is the
    cosine similarity between the embeddings of the query and memory.
    *Importance* is a rating the language model assigns when the memory is
    created. The paper uses a prompt like this one.

    ```
    On the scale of 1 to 10, where 1 is purely mundane (e.g., brushing teeth, mak
    ing bed) and 10 is extremely poignant (e.g., a break up, college acceptance),
     rate the likely poignancy of the following piece of memory.
    
    Memory: buying groceries at The Willows Market and Pharmacy
    Rating: <fill in>
    ```

    The paper min-max scales each of the three scores across the memory stream
    and adds them with equal weights. The highest-ranked memories that fit in
    the context window go into the prompt [@genagents, sec. 4.1].


![**Figure 6 of ** [@genagents]. The memory stream consists of a large number of observations that are relevant and irrelevant to the agent's current situation. Retrieval identifies a subset of these observations that should be passed to the language model to condition its
response to the situation.](../images/post/ai-agents-foundations-2cc681c996274efea961a680d70e04c9.png)


**Reflection.** 
Generative agents, when equipped with only raw observational memory, struggle to generalize or make inferences.
Reflections are introduced as a second type of memory (e.g. the retrieval function also returns reflections). 
Reflections are higher-level, more abstract thoughts
generated by the agent. Because they are a type of memory, they
are included alongside other observations when retrieval occurs. In the paper, this triggers for an agent
when the sum of importance scores exceed 150. This is done as follows:

1. **Three points of reflection.** Query the 100 most recent memory. Prompt the language model: "Given only the information above, what are 3 most salient high-level questions we can answer about the subjects in the statements?". This method directly extracts specific lines of inquiry rather than producing a general summary.

2. **Reflection proper.** For one question $Q$ at a time, retrieve memories using the retrieval function $f(Q).$ Prompt the language model to extract insights:
    
    ```text
    Statements about Klaus Mueller
    1. Klaus Mueller is writing a research paper
    2. Klaus Mueller enjoys reading a book
    on gentrification
    3. Klaus Mueller is conversing with Ayesha Khan
    about exercising [...]

    What 5 high-level insights can you infer from the above statements? 
    (example format: insight (because of 1, 5, 3))
    ```

3. **Extend reflection tree.** Store these in the memory stream and cite the particular records that served as evidence for the insights. Since reflections are memories, agents can reflect on past reflections.
As a result, agents generate trees of reflections (@fig-reflections-tree): the [leaf nodes]{.underline} of
the tree represent the base observations, and the non-leaf nodes
represent [thoughts]{.underline} that become more abstract and higher-level the
higher up the tree they are. Hence, we can assign a **depth** value to each memory.

![**Figure 7 of ** [@genagents] A reflection tree for Klaus Mueller. The agent’s observations of the world, represented in the leaf nodes, are recursively synthesized to derive Klaus’s self-notion that he is highly dedicated to his research.](../images/post/ai-agents-foundations-de9eeed0ae8949f185707fdd327d69a0.png){#fig-reflections-tree}


**Planning and reacting.** Agents need to plan over a longer time horizon to ensure that their sequence
of actions is coherent and believable. A **plan** includes a location, a starting time, and a duration.
Like reflections, plans are [stored in the memory stream]{.underline} and are included in the retrieval process. This
allows the agent to consider observations, reflections, and plans all together when deciding how to behave. 

First, at the start of each day, the language model is prompted with the agent's summary description (e.g. name, traits, and a summary of their recent experiences) and a summary of their previous day. A general plan for the day's agenda is obtained (7-8 items). These are then recursively decomposed to create finer-grained actions, first into hour-long chunks of actions, and then to 5-15 minute chunks (@fig-genagents-plan).

Agents may [change plans]{.underline} mid-stream if needed given observations from their environment. We prompt the language model with these observations to decide whether the agent should continue with their existing plan, or **react**. The prompt template is given by:
```text
[Agent's Summary Description:] 
{timestamp,agent_current_state}

[Observation:] 
{observation} 

[Summary of relevant context from John’s memory:] 
{context_summary}

Should John react to the observation, and if so, 
what would be an appropriate reaction?
``` 
Let `O` be the observer and `E` be the observed entity. The context is generated through two prompts that retrieve memories by summarizing the combined output of the two queries with the retrieval function: `"What is [O]'s relationship with [E]?"` and `"[E] is [status_of_E]"`.
Note that these include plans, since plans are also in the memory stream.
We then regenerate the agent's existing plan starting from the time when the reaction takes place. Finally, if the output action indicates an interaction between agents, we generate their dialogue.

![**Planned tasks of Yuriko Yamamoto.** Current status: Yuriko Yamamoto is working on a tax compliance project for a local business. She is also taking classes to stay up to date on new tax laws. Yuriko is also curious about who will be running for the local mayor election next month.](../images/post/ai-agents-foundations-532d9894c4b34b6682ed5746be1031e6.png){#fig-genagents-plan}

:::{.callout-note}
**Generative Agents** [@genagents] showed that LLMs can simulate believable behavior by having (1) a **seeded agent identity**, and (2) a **core memory module** that consist of observations, reflections, and current plans. Since memories are noisy and will quickly fill LLM context, having a **retrieval function** is crucial given the query that applies to the current situation. 
This is augmented by having **reflections** which are *abstract* observations that *compress* previous events and that were recollected based on recent events. 
See Appendix B of the paper where agents are interviewed (e.g. "Give an introduction of yourself") which can be answered using the retrieval function.
Finally, allowing LLMs to interact results in **emergent phenomenon** (i.e. occurences that not explicitly built into the system).

:::

## Code experiments


### Exercise 1. Few-shot chain-of-thought prompting


Does showing the intermediate steps in a few worked examples change
how often the model solves a new problem? Following the comparison in
[@cot, sec. 3], both prompts use the same example questions and final
answers. The chain-of-thought prompt includes the intermediate steps
for those answers.


Use the first ten problems from the [GSM8K test split](https://github.com/openai/grade-school-math)
and the same model and final-answer format in both conditions. The two
demonstrations below are written separately from the test problems.
The grader reads only the final `Answer:` line and compares its value
numerically, so `12` and `12.0` count as the same answer.


In [5]:
import os

from openai import OpenAI

MODEL_NAME = "deepseek-flash"
BASE_URL = "https://api.deepseek.com"
API_KEY = os.environ["DEEPSEEK_API_KEY"]
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def chat(messages, model=MODEL_NAME, max_tokens=512):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ""

In [ ]:
import json
import re
from decimal import Decimal

import requests
from IPython.display import Markdown, display



test_url = (
    "https://raw.githubusercontent.com/openai/grade-school-math/"
    "master/grade_school_math/data/test.jsonl"
)
response = requests.get(test_url, timeout=15)
response.raise_for_status()
problems = [json.loads(line) for line in response.text.splitlines()[:10]]

EXAMPLES = [
    (
        "A library lends 18 books on Monday and twice as many on Tuesday. "
        "How many books does it lend across both days?",
        "On Tuesday it lends 2 × 18 = 36 books. The total is 18 + 36 = 54.",
        54,
    ),
    (
        "Nia buys three boxes of 12 pencils and gives away 8 pencils. "
        "How many pencils does she have left?",
        "The boxes contain 3 × 12 = 36 pencils. After giving away 8, "
        "she has 36 − 8 = 28.",
        28,
    ),
]


def math_messages(question, show_steps):
    messages = [{
        "role": "system",
        "content": "Solve the problem. End with a line of the form Answer: <number>.",
    }]
    for example_question, steps, answer in EXAMPLES:
        final_line = f"Answer: {answer}"
        reply = f"{steps}\n{final_line}" if show_steps else final_line
        messages.extend([
            {"role": "user", "content": example_question},
            {"role": "assistant", "content": reply},
        ])
    messages.append({"role": "user", "content": question})
    return messages


def final_number(reply):
    lines = reply.strip().splitlines()
    if not lines:
        return None
    match = re.fullmatch(
        r"Answer:[ \t]*([+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\.?",
        lines[-1],
        flags=re.IGNORECASE,
    )
    if match:
        return Decimal(match[1].replace(",", ""))


Send both prompts for each problem and keep the raw responses alongside
their parsed answers. A response without a parseable final line counts
as unsuccessful, and the results report format failures separately.


In [ ]:
math_results = []

for problem in problems:
    question = problem["question"]
    reference = Decimal(problem["answer"].rsplit("####", 1)[1].strip().replace(",", ""))
    direct = await chat(math_messages(question, show_steps=False))
    cot = await chat(math_messages(question, show_steps=True))
    math_results.append({
        "question": question,
        "reference": reference,
        "direct": final_number(direct),
        "cot": final_number(cot),
        "direct_reply": direct,
        "cot_reply": cot,
    })


**Results.** Compare the two answers for each problem. Inspect the full
responses for the first problem where the parsed answers differ, or for
the first problem if they all agree.


In [ ]:
#| code-fold: true
def show_table(headers, rows):
    def row_line(row):
        fields = [" ".join(str(value).split()).replace("|", "&#124;") for value in row]
        return "| " + " | ".join(fields) + " |"

    lines = [row_line(headers), row_line(["---"] * len(headers))]
    lines.extend(row_line(row) for row in rows)
    display(Markdown("\n".join(lines)))


show_table(
    ["Problem", "Reference", "Direct", "Chain of thought"],
    [
        (index, row["reference"], row["direct"] if row["direct"] is not None else "Unparsed",
         row["cot"] if row["cot"] is not None else "Unparsed")
        for index, row in enumerate(math_results, start=1)
    ],
)
for key, name in (("direct", "Direct"), ("cot", "Chain of thought")):
    correct = sum(row[key] == row["reference"] for row in math_results)
    unparsed = sum(row[key] is None for row in math_results)
    print(f"{name} accuracy: {correct}/{len(math_results)} ({unparsed} unparsed)")

example_index = next(
    (i for i, row in enumerate(math_results) if row["direct"] != row["cot"]),
    0,
)
example = math_results[example_index]
print(f"\nProblem {example_index + 1}\n{example['question']}")
print(f"\nDirect response\n{example['direct_reply']}")
print(f"\nChain-of-thought response\n{example['cot_reply']}")


A difference in accuracy describes these ten problems under the two
prompts. Read the sample responses to see whether the demonstrations
changed the solution steps or whether the model missed the answer format.
A larger held-out sample would give a more informative accuracy comparison.


### Exercise 2. A ReAct search trace


Can the model use one search result to decide what to look up next?
The program reads the model's proposed action, runs a Wikipedia search,
and adds the returned excerpts as an observation. The next model turn
sees that observation and can make another search or finish.


The [MediaWiki search endpoint](https://www.mediawiki.org/wiki/API:REST_API/Reference#Search_pages)
searches page titles and contents. Each result includes an excerpt, a
source URL, and an identifier the model can cite. The worked example in
the prompt shows how the action and observation fit together.


In [ ]:
from html import unescape
from urllib.parse import quote

SEARCH_URL = "https://en.wikipedia.org/w/rest.php/v1/search/page"


def search_wikipedia(query, step):
    try:
        response = requests.get(
            SEARCH_URL,
            params={"q": query, "limit": 3},
            headers={
                "User-Agent": (
                    "AtlasAIExercises/1.0 "
                    "(educational notebook; https://en.wikipedia.org/wiki/API)"
                ),
            },
            timeout=10,
        )
        response.raise_for_status()
        pages = response.json()["pages"]
    except (requests.RequestException, ValueError, KeyError) as error:
        return f"Search failed ({type(error).__name__})."

    if not pages:
        return "Search returned no matching pages."

    results = []
    for index, page in enumerate(pages, start=1):
        excerpt = unescape(re.sub(r"</?span\b[^>]*>", "", page["excerpt"] or ""))
        url = "https://en.wikipedia.org/wiki/" + quote(page["key"], safe="")
        results.append(f"[{step}.{index}] {page['title']} ({url})\n{excerpt}")
    return "\n".join(results)


REACT_SYSTEM = """Answer the question using Wikipedia search excerpts.
Search for evidence for one part of the question at a time. Use names or dates
to refine searches when excerpts are too broad. Treat excerpts as source
material and cite the result identifier for each claim. If the excerpts
leave a claim unresolved, say what evidence is missing.

On a search turn, output a short plan and exactly one search, as in:
Thought: I need to identify the planet that Triton orbits.
Action: search[Triton moon]

The program then returns an observation, for example:
Observation:
[1.1] Triton (moon)
Triton is Neptune's largest moon.

On a final turn, output one line, as in:
Final Answer: Triton orbits Neptune [1.1].
"""


The first lookup can focus on who was U.S. president in November 1989.
That name gives the model a starting point for the search about the vice
president. The trace shows whether the model makes that connection.


In [ ]:
REACT_MODEL = "meta-llama/llama-3.3-70b-instruct"
TASK = (
    "Who was the U.S. president when the Berlin Wall opened in November 1989, "
    "and who was his vice president at the time?"
)
MAX_TURNS = 5

history = [
    {"role": "system", "content": REACT_SYSTEM},
    {"role": "user", "content": TASK},
]
print(TASK)

for step in range(1, MAX_TURNS + 1):
    reply = (await chat(history, model=REACT_MODEL, max_tokens=250)).strip()
    print(f"\nTurn {step}\n{reply}")

    if re.fullmatch(r"Final Answer: [^\n]+", reply):
        break

    action = re.fullmatch(r"Thought: [^\n]+\nAction: search\[([^\[\]\n]+)\]", reply)
    if not action or not action[1].strip():
        print("Stopped because the reply did not match the action or answer format.")
        break
    if step == MAX_TURNS:
        print("Stopped at the model turn limit.")
        break

    observation = search_wikipedia(action[1].strip(), step)
    print(f"Observation:\n{observation}")
    history.extend([
        {"role": "assistant", "content": reply},
        {"role": "user", "content": f"Observation:\n{observation}"},
    ])


The expected names are George H. W. Bush and Dan Quayle. Check the cited
excerpts for both claims. If an excerpt identifies Bush but says nothing
about his vice president, it supports only the first part. A failed search
can still be useful if the next turn uses the failure to revise its query.

[Office of the Historian](https://history.state.gov/milestones/1989-1992/fall-of-communism)
· [U.S. Senate list of vice presidents](https://www.senate.gov/about/officers-staff/vice-president/vice-presidents.htm)


::: {.callout-note}
In the ReAct paper's review of sampled HotpotQA failures, the authors
attributed 0% of ReAct failures and 56% of CoT failures to hallucination.
They also found hallucinated facts or reasoning in some ReAct traces
with correct final answers [@ReAct2023, tbl. 2]. The excerpts in this
exercise let readers check whether each claim follows from the material
the model received.
:::


### Exercise 3. Answering with and without evidence


Does supplying a relevant passage change the answer? For each question,
give the same model one request with an empty evidence field and one
request with two short passages. Use the same question and instructions
in both requests.


The passages are fixed paraphrases of the linked sources. Each pair
contains one passage that answers the question and one related passage.
The reference answer and supporting passage number are used for grading
after the response.


In [ ]:
QUESTIONS = [
    {
        "question": "What is the capital of Comoros?",
        "answer": "Moroni",
        "passages": (
            "Moroni is the capital of Comoros.",
            "The Comorian franc is the country's national currency.",
        ),
        "support": "1",
        "source": "https://data.un.org/legacy/en/iso/km.html",
    },
    {
        "question": "What tissue makes up a shark's skeleton?",
        "answer": "cartilage",
        "passages": (
            "Sharks use gills to take up oxygen from water.",
            "A shark's skeleton is made of cartilage.",
        ),
        "support": "2",
        "source": "https://www.fisheries.noaa.gov/feature-story/12-shark-facts-may-surprise-you",
    },
    {
        "question": "On what date did NASA's Dawn enter orbit around Ceres? Give YYYY-MM-DD.",
        "answer": "2015-03-06",
        "passages": (
            "The Dawn spacecraft orbited Vesta before it arrived at Ceres.",
            "Dawn entered orbit around Ceres on March 6, 2015.",
        ),
        "support": "2",
        "source": "https://science.nasa.gov/dwarf-planets/ceres/exploration/",
    },
    {
        "question": (
            "Approximately how far is Proxima Centauri from Earth in light-years? "
            "Give only the number, to two decimal places."
        ),
        "answer": "4.24",
        "passages": (
            "Proxima Centauri is roughly 4.24 light-years from Earth.",
            "A light-year measures the distance light travels in one year.",
        ),
        "support": "1",
        "source": "https://science.nasa.gov/mission/voyager/voyager-1/voyager-1-what-is-a-light-day/",
    },
    {
        "question": "What is the primary programming language of the Linux kernel?",
        "answer": "C",
        "passages": (
            "The Linux kernel supports Rust when the CONFIG_RUST option is enabled.",
            "C is the primary language used to write the Linux kernel.",
        ),
        "support": "2",
        "source": "https://www.kernel.org/doc/html/latest/process/programming-language.html",
    },
]


Ask for a short answer and the number of the passage that supports it.
The model can also report that it has no supporting passage. Both
conditions use the same response format.


In [ ]:
EVIDENCE_SYSTEM = """Give the shortest factual answer in the format requested.
Use the supplied evidence if it answers the question. You may answer from
your own knowledge if evidence is missing. Use UNKNOWN if unsure.
Return two lines:
Answer: <answer>
Evidence: <1, 2, or NONE if no passage supports the answer>
"""


async def answer_question(question, evidence):
    reply = await chat([
        {"role": "system", "content": EVIDENCE_SYSTEM},
        {"role": "user", "content": f"Question: {question}\nEvidence:\n{evidence}"},
    ], max_tokens=120)
    match = re.fullmatch(
        r"Answer: ([^\n]+)\nEvidence: (1|2|NONE)", reply.strip(), flags=re.IGNORECASE,
    )
    if not match:
        return reply or "(empty response)", None
    return match[1].strip(), match[2].upper()


Run the pair with an empty evidence field and then with both numbered
passages. Each request starts with a fresh message list, so the answer
from the first condition is outside the context of the second.


In [ ]:
evidence_results = []

for item in QUESTIONS:
    passages = "\n".join(
        f"[{index}] {passage}" for index, passage in enumerate(item["passages"], start=1)
    )
    without_evidence = await answer_question(item["question"], "")
    with_evidence = await answer_question(item["question"], passages)
    evidence_results.append((item, without_evidence, with_evidence))


A correct answer and a supporting citation are separate observations.
A response can get the answer right using its own knowledge. It can
also identify the relevant passage and still give the wrong answer.


**Results.** Compare short answers with the reference and check whether
the evidence condition cites the expected passage number. The passages
below the table let you inspect that judgment for each question.


In [ ]:
#| code-fold: true
def describe_answer(result, reference):
    answer, citation = result
    if citation is None:
        return f"{answer} (format error)"
    correct = answer.rstrip(".").casefold() == reference.casefold()
    verdict = "correct" if correct else "incorrect"
    return f"{answer} ({verdict}, evidence {citation})"


show_table(
    ["Case", "Reference", "No evidence", "With evidence", "Cites relevant passage"],
    [
        (index, item["answer"], describe_answer(without, item["answer"]),
         describe_answer(with_, item["answer"]),
         "Yes" if with_[1] == item["support"] else "No")
        for index, (item, without, with_) in enumerate(evidence_results, start=1)
    ],
)

case_notes = []
for index, item in enumerate(QUESTIONS, start=1):
    first, second = item["passages"]
    case_notes.append(
        f"**{index}. {item['question']}**\n\n"
        f"- (1) {first}\n"
        f"- (2) {second}\n\n"
        f"[Source]({item['source']})"
    )
display(Markdown("\n\n".join(case_notes)))


Look for a question where the answer changes when the passage is present.
If both answers are correct, the passage can still make the answer easier
to verify. These five questions illustrate how to inspect evidence use
in individual answers.


In a live retrieval system, the search step may return only related
passages. The model then has to recognize that the context is insufficient,
reformulate its query, or report that it could not verify the answer.


### Exercise 4. A generative agent’s memory stream {#sec-agent-memory}

Mara is organizing a repair café in a simulated town. She needs a venue
after the town hall closes. Her memory stream contains an earlier plan,
observations about the town, and a reflection about past events.
Which two memories would help her decide where to set up?

This example uses the retrieval rule from [@genagents, sec. 4.1]. The
importance ratings and topic vectors are assigned by hand so the demo
runs locally. The three vector entries mark whether a memory concerns
the event, the venue, or daily routine. In the paper, the language
model rates importance and relevance uses text embeddings.


In [ ]:
from dataclasses import dataclass, field
from math import hypot


@dataclass
class Memory:
    id: str
    kind: str
    text: str
    created_at: int
    importance: int
    topics: tuple[int, int, int]
    last_accessed: int = field(init=False)

    def __post_init__(self):
        self.last_accessed = self.created_at


# Times are hours since the simulation began.
memories = [
    Memory("M1", "Plan", "Set up the repair café at the town hall.", 5, 7, (1, 1, 0)),
    Memory(
        "M2", "Observation",
        "The librarian offered a meeting room as a backup for the repair café.",
        6, 9, (1, 1, 0),
    ),
    Memory(
        "M3", "Reflection",
        "A quick demonstration helps newcomers use the tools.",
        18, 4, (1, 0, 0),
    ),
    Memory(
        "M4", "Observation",
        "The town hall is closed because of a roof leak.",
        19, 10, (0, 1, 0),
    ),
    Memory("M5", "Observation", "Mara made tea.", 20, 1, (0, 0, 1)),
]


def min_max(values):
    low, high = min(values), max(values)
    if high == low:
        return [0.0] * len(values)
    return [(value - low) / (high - low) for value in values]


def cosine_similarity(a, b):
    dot = sum(left * right for left, right in zip(a, b, strict=True))
    return dot / (hypot(*a) * hypot(*b))


def rank_memories(stream, query_topics, now, weights=(1, 1, 1)):
    recency = min_max([0.995 ** (now - memory.last_accessed) for memory in stream])
    importance = min_max([memory.importance for memory in stream])
    relevance = min_max([
        cosine_similarity(memory.topics, query_topics) for memory in stream
    ])
    scores = zip(stream, recency, importance, relevance, strict=True)

    def weighted_score(row):
        return sum(
            weight * value for weight, value in zip(weights, row[1:], strict=True)
        )

    return sorted(scores, key=weighted_score, reverse=True)


The query concerns the event and its venue, so its topic vector is
`(1, 1, 0)`. Compare retrieval by last access alone with retrieval
using all three scores. A limit of two memories stands in for a
context budget. Once a memory enters the prompt, record the current
hour as its last access.


In [2]:
from IPython.display import Markdown, display

now = 21
question = "Where should Mara set up the repair café?"
ranked = rank_memories(memories, query_topics=(1, 1, 0), now=now)
recent = sorted(memories, key=lambda memory: memory.last_accessed, reverse=True)[:2]
selected = [memory for memory, *_ in ranked[:2]]

table = [
    "| ID | Memory | Recency | Importance | Relevance | Total |",
    "| --- | --- | ---: | ---: | ---: | ---: |",
]
for memory, recency, importance, relevance in ranked:
    total = recency + importance + relevance
    table.append(
        f"| {memory.id} | {memory.kind}. {memory.text} | {recency:.2f} | {importance:.2f} | "
        f"{relevance:.2f} | {total:.2f} |"
    )
display(Markdown("\n".join(table)))

print("Two most recently accessed memories")
for memory in recent:
    print(f"- {memory.text}")

print("\nPrompt using the two highest-scoring memories")
print("You are Mara, the organizer of the repair café.")
for memory in selected:
    print(f"- {memory.text}")
print(question)

print("\nLast access after retrieval")
for memory in selected:
    previous_access = memory.last_accessed
    memory.last_accessed = now
    print(f"- Hour {previous_access} → {memory.last_accessed}. {memory.text}")


| ID | Memory | Recency | Importance | Relevance | Total |
| --- | --- | ---: | ---: | ---: | ---: |
| M4 | Observation. The town hall is closed because of a roof leak. | 0.93 | 1.00 | 0.71 | 2.64 |
| M2 | Observation. The librarian offered a meeting room as a backup for the repair café. | 0.06 | 0.89 | 1.00 | 1.95 |
| M3 | Reflection. A quick demonstration helps newcomers use the tools. | 0.86 | 0.33 | 0.71 | 1.90 |
| M1 | Plan. Set up the repair café at the town hall. | 0.00 | 0.67 | 1.00 | 1.67 |
| M5 | Observation. Mara made tea. | 1.00 | 0.00 | 0.00 | 1.00 |

Two most recently accessed memories
- Mara made tea.
- The town hall is closed because of a roof leak.

Prompt using the two highest-scoring memories
You are Mara, the organizer of the repair café.
- The town hall is closed because of a roof leak.
- The librarian offered a meeting room as a backup for the repair café.
Where should Mara set up the repair café?

Last access after retrieval
- Hour 19 → 21. The town hall is closed because of a roof leak.
- Hour 6 → 21. The librarian offered a meeting room as a backup for the repair café.


Both approaches surface the closure. The three-score ranking also
brings back the librarian’s earlier offer, which gives Mara a specific
alternative. The tea observation was recent but had little to do with
the venue. Retrieving the offer also refreshes its last-access time, so
it can receive a higher recency score on a later lookup.


#### Experiment. Comparing memory retrieval rules {#sec-agent-memory-experiment}

Does combining the three scores help Mara retrieve the memories she
needs? Use the same five memories and two slots for each question.
Restore last-access times to their initial values so the earlier
walkthrough cannot affect the comparison. The only change between
methods is which scores receive weight. Ties follow the order of the
memory stream.

Label the supporting memories for each question before ranking them.
Recall at two is the share of those memories found in the two results.
The mean gives each question equal weight.


In [3]:
from dataclasses import replace
from statistics import mean

# A new Memory starts with last_accessed equal to created_at.
initial_stream = [replace(memory) for memory in memories]
cases = [
    ("Where should Mara set up now?", (1, 1, 0), {"M2", "M4"}),
    ("What helps newcomers use the tools?", (1, 0, 0), {"M3"}),
    ("What did Mara make most recently?", (0, 0, 1), {"M5"}),
    ("What was Mara’s original venue plan?", (1, 1, 0), {"M1"}),
]
weights_by_method = {
    "Recency only": (1, 0, 0),
    "Relevance only": (0, 0, 1),
    "Combined": (1, 1, 1),
}
recalls = {method: [] for method in weights_by_method}

results_table = [
    "| Question | Supporting memories | Recency only | Relevance only | Combined |",
    "| --- | --- | --- | --- | --- |",
]
for prompt, topics, supporting_ids in cases:
    row = [prompt, ", ".join(sorted(supporting_ids))]
    for method, weights in weights_by_method.items():
        ranked = rank_memories(initial_stream, topics, now, weights=weights)
        retrieved_ids = [memory.id for memory, *_ in ranked[:2]]
        hits = len(supporting_ids.intersection(retrieved_ids))
        recalls[method].append(hits / len(supporting_ids))
        row.append(f"{', '.join(retrieved_ids)} ({hits}/{len(supporting_ids)})")
    results_table.append("| " + " | ".join(row) + " |")

averages = [f"{mean(scores):.1%}" for scores in recalls.values()]
results_table.append("| Mean recall at two | | " + " | ".join(averages) + " |")
display(Markdown("\n".join(results_table)))


| Question | Supporting memories | Recency only | Relevance only | Combined |
| --- | --- | --- | --- | --- |
| Where should Mara set up now? | M2, M4 | M5, M4 (1/2) | M1, M2 (1/2) | M4, M2 (2/2) |
| What helps newcomers use the tools? | M3 | M5, M4 (0/1) | M3, M1 (1/1) | M3, M4 (1/1) |
| What did Mara make most recently? | M5 | M5, M4 (1/1) | M5, M1 (1/1) | M5, M4 (1/1) |
| What was Mara’s original venue plan? | M1 | M5, M4 (0/1) | M1, M2 (1/1) | M4, M2 (0/1) |
| Mean recall at two | | 37.5% | 87.5% | 75.0% |

The combined score retrieves both memories needed for the venue
change. Relevance alone gets the library offer and the original plan,
leaving out the closure. Across these four questions, relevance alone
has higher mean recall because it also retrieves the original plan
when asked for it. The two venue questions share a topic vector, so
the combined ranker gives both questions the same order even though
they need different memories.

This measures whether supporting memories reach the prompt. The
ratings, topic vectors, and supporting labels are hand-authored for
this town, so the averages describe these four cases.


---

&#9632;